In [ ]:
from IPython.display import HTML
HTML(open('../style.css', 'r').read())

In [ ]:
from typing import TypeVar

# Minimizing a Finite State Machine

In this notebook we show how a <span style="font-variant:small-caps;">Dfa</span> can be minimized by identifying states that are equivalent.

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright 07-Minimize.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

## Some Type Definitions

* `Char` is an alias for `str`.
* `State` is the abstract type of the states that make up the <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
Char  = str
State = TypeVar('State')

`TransRel` is the type that describes the transition function $\delta$.

In [ ]:
TransRel = dict[tuple[State, Char], State]

`DFA` is the type that describes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
DFA = tuple[set[State], set[Char], TransRel, State, set[State]]

`Pair` is a pair of states.

In [ ]:
Pair = tuple[State, State]

The states of the minimized <span style="font-variant:small-caps;">Dfa</span> will be  sets of states of the original <span style="font-variant:small-caps;">Dfa</span>.  These states will contain those states that are *equivalent*.

In [ ]:
SetState = frozenset[State]

In [ ]:
TransRelSet = dict[tuple[SetState, Char], SetState]

In [ ]:
Min_DFA = tuple[set[SetState], set[Char], TransRelSet, SetState, set[SetState]]

The function `arb(M)` takes a non-empty set `M` as its argument and returns an *arbitrary* element from this set.  The set `M` is not changed.  The function `arb`also works if `M` is a `frozenset`. 

In order to type-check this function we define an abstract type `S` that serves as a *type parameter* and  denotes the type of the elements of the set `M`.

**Function `arb(M)`**
- *Input:* `M` is a non-empty set or frozenset.
- *Output:* An arbitrary element of `M`.

In [ ]:
def arb[S](M: set[S] | frozenset[S]) -> S:
    x, *_ = M
    return x

The function `cart_prod(A, B)` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where $A \times B$ is defined as follows:
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$
In order to denote the type of the second set, we define the abstract type `T`.

**Function `cart_prod(A, B)`**
- *Input:* `A` and `B` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
def cart_prod[S, T](A: set[S], B: set[T]) -> set[tuple[S, T]]:
    return { (x, y) for x in A for y in B }

In [ ]:
cart_prod({1, 2}, {'a', 'b', 'c'})

The function `separate` takes four arguments:
- `Pairs`  is a set of pairs of states from some given <span style="font-variant:small-caps;">Dfa</span> $F$.

   If $(p_1, p_2) \in \texttt{Pairs}$, then $p_1$ and $p_2$ are known to be *separable*.
   
   Two states $p_1$ and $p_2$ are *separable* if there is a string $w$ such that processing $w$ in state
   $p_1$ leads to an accepting state, while processing $w$ in state $p_2$ leads to a state that is not 
   accepting, or vice versa.
- `States` is the set of all states of the <span style="font-variant:small-caps;">Dfa</span> $F$,
- `Σ` is the alphabet of the <span style="font-variant:small-caps;">Dfa</span> $F$.
- `𝛿` is the transition function of the <span style="font-variant:small-caps;">Dfa</span>

The function `separate(Pairs, States, Σ, 𝛿)` computes the set of pairs of states $(q_1, q_2)$ that are *separable* because there is some character $c \in \Sigma$ such that 
$$\delta(q_1,c) = p_1, \quad \textrm{but} \quad \delta(q_2,c) = p_2 $$
and $(p_1, p_2)$ is already known to be *separable* because $(p_1, p_2) \in \textrm{Pairs}$.

**Function `separate(Pairs, States, Σ, 𝛿)`**
- *Input:* `Pairs` is a set of pairs of states that are known to be separable, `States` is the set of states of a <span style="font-variant:small-caps;">Dfa</span>, `Σ` is its alphabet, and `𝛿` is its transition function.
- *Output:* The set of all pairs $(q_1, q_2)$ such that $(\delta(q_1, c), \delta(q_2, c)) \in \texttt{Pairs}$ for some character $c$.

In [ ]:
def separate(Pairs: set[Pair], States: set[State], Σ: set[Char], 𝛿: TransRel) -> set[Pair]:
    Result = { (q1, q2) for q1 in States
                        for q2 in States
                        for c  in Σ 
                        if (𝛿[q1, c], 𝛿[q2, c]) in Pairs
             }
    return Result

Given a state `p` and a `Partition` of the set of all states, the function `find_equivalence_class(p, Partition)` returns the equivalence class of `p`, i.e. it returns the set of states from `Partition` that contains the state `p`. 

**Function `find_equivalence_class(p, Partition)`**
- *Input:* `p` is a state and `Partition` is a partition of the set of states.
- *Output:* The set in `Partition` that contains `p`.

In [ ]:
def find_equivalence_class(p: State, Partition: set[frozenset[State]]) -> frozenset[State]:
    return arb({ C for C in Partition if p in C })

The function `all_separable(Q, A, Σ, 𝛿)` takes four arguments:
* `Q` is the set of states of the <span style="font-variant:small-caps;">Dfa</span>.
* `A` is the set of all accepting states,
* `Σ`  is the alphabet.
* `𝛿` is the transition function.  

  `𝛿` is represented as a dictionary.   

The function computes the set of all Pairs `(p, q)` such that `p` and `q` are separable, i.e. all pairs such that
$$ \exists s \in \Sigma^*: \bigl(\delta^*(p, s) \in A \wedge \delta^*(q,s) \not\in A\bigr) \vee 
                           \bigl(\delta^*(p, s) \not\in A \wedge \delta^*(q,s) \in A\bigr). 
$$
The result is computed via a *fixed point* computation:
1. Initially, all accepting states are separated from the non-accepting states.
2. If $p_1$ and $p_2$ are separable and $q_1$ and $q_2$ are two states such that there is a character 
   $c \in \Sigma$ such that
   $$ \delta(q_1, c) = p_1 \quad \mbox{and} \quad \delta(q_2, c) = p_2, $$
   then $q_1$ and $q_2$ are separable.

**Function `all_separable(Q, A, Σ, 𝛿)`**
- *Input:* `Q` is the set of states of a complete <span style="font-variant:small-caps;">Dfa</span>, `A` is its set of accepting states, `Σ` is its alphabet, and `𝛿` is its transition function.
- *Output:* The set of all pairs of separable states.

In [ ]:
def all_separable(Q: set[State], A: set[State], Σ: set[Char], 𝛿: TransRel) -> set[Pair]:
        Separable = cart_prod(Q - A, A) | cart_prod(A, Q - A)
        while True:
            NewPairs = separate(Separable, Q, Σ, 𝛿)
            if NewPairs <= Separable:
                return Separable
            Separable |= NewPairs

The function `reachable(q0, Σ, 𝛿)` takes three arguments:
* `q0` is the start state of a <span style="font-variant:small-caps;">Dfa</span>,
* `Σ`  is the alphabet.
* `𝛿`  is the transition function.  The transition function is assumed to be *complete*. `𝛿` is represented as a dictionary.   

It returns the set of all states that can be reached from the start state `q0` by reading strings of characters from `Σ`.

The result is computed via a *fixed point* computation.

**Function `reachable(q0, Σ, 𝛿)`**
- *Input:* `q0` is the start state of a complete <span style="font-variant:small-caps;">Dfa</span>, `Σ` is its alphabet, and `𝛿` is its transition function.
- *Output:* The set of all states that can be reached from `q0`.

In [ ]:
def reachable(q0: State, Σ: set[str], 𝛿: TransRel) -> set[State]:
    Result = { q0 }
    while True:
        NewStates = { 𝛿[p, c] for p in Result for c in Σ }
        if NewStates <= Result:
            return Result
        Result |= NewStates

The function `minimize(F)` takes a <span style="font-variant:small-caps;">Dfa</span> `F` as its input.
Here `F` is a 5-tuple of the form
$$ F = (Q, \Sigma, \delta, q_0, A). $$
The <span style="font-variant:small-caps;">Dfa</span> $F$ has to be *complete*, i.e. $\delta(q, c)$ has to be defined for all states $q$ and all
characters $c$.
The algorithm performs the following steps:
1. All unreachable states are eliminated.
2. All accepting states are separated from all non-accepting states.
3. States are separated as long as possible.
   Two states $p_1$ and $p_2$ are separable if there is a string 
   $w \in \Sigma^*$ such that 
   $$ \Bigl(\delta^*(p_1,w) \in A \;\wedge\; \delta^*(p_2,w) \not\in A\Bigr) \quad \vee \quad 
      \Bigl(\delta^*(p_1,w) \not\in A \;\wedge\; \delta^*(p_2,w) \in A\Bigr)
   $$
4. States that are not separable are *equivalent* and are therefore identified and grouped
   in equivalence classes.  The states in an equivalence class are then identified.

**Function `minimize(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> with the minimal number of states that accepts the same language as `F`.  Its states are sets of equivalent states of `F`.

In [ ]:
def minimize(F: DFA) -> Min_DFA:
    Q, Σ, 𝛿, q0, A = F
    Q              = reachable(q0, Σ, 𝛿)
    Separable      = all_separable(Q, A, Σ, 𝛿)
    Equivalent     = cart_prod(Q, Q) - Separable
    EquivClasses   = { frozenset({ p for p in Q if (p, q) in Equivalent })
                       for q in Q
                     }
    newQ0          = arb({ M for M in EquivClasses if q0 in M })
    newAccept      = { M for M in EquivClasses if arb(M) in A }   
    newDelta: TransRelSet = {}
    for q in Q:
        classOfQ = find_equivalence_class(q, EquivClasses)
        for c in Σ:
            newDelta[(classOfQ, c)] = find_equivalence_class(𝛿[q, c], EquivClasses)
    return EquivClasses, Σ, newDelta, newQ0, newAccept